# Test Data Scientist pour banquier

Test de manipulation de données (Pandas / SQL)

# 1. Objectif pour le candidat.

Construire un modèle de scoring interprétable et robuste permettant d’estimer le risque de défaut d’un client.  

Livrables attendus :  
* Un notebook clair (.ipynb)
* Un modèle entraîné
* Une évaluation propre
* Des graphiques explicatifs
* Des commentaires métier

# 2. Données

**clients.csv**

| colonne           | description                     |
| ----------------- | ------------------------------- |
| client_id         | identifiant                     |
| age               | âge                             |
| income            | revenu mensuel                  |
| employment_status | CDI, CDD, freelance, unemployed |
| seniority_years   | ancienneté professionnelle      |
| gender            | M / F                           |
| region            | région                          |


**loans.csv**

| colonne           | description               |
| ----------------- | ------------------------- |
| client_id         | identifiant               |
| loan_amount       | montant du prêt           |
| duration_months   | durée                     |
| interest_rate     | taux                      |
| previous_defaults | nombre d'incidents passés |
| default           | target                    |

**transactions.csv**

| colonne      | description                           |
| ------------ | ------------------------------------- |
| client_id    | identifiant                           |
| month        | mois relatif (−6 à +6 autour du prêt) |
| balance      | solde mensuel                         |
| overdraft    | découvert utilisé                     |
| late_payment | 0/1                                   |


⚠️ Le dataset contient volontairement:  
* valeurs manquantes
* variables corrélées
* fuite de données subtile (transactions postérieures au défaut)
* variable proxy d’âge
*déséquilibre de classes (~15% défauts)

## 3. Consignes données au candidat :

**Partie A – Exploration & nettoyage**. 

* Charger les données
* Identifier les problèmes de qualité
* Joindre correctement les tables
* Créer au moins 5 features pertinentes
* Justifier les choix. 

**Partie B – Modélisation**
* Entraîner au minimum :
une régression logistique
un modèle non-linéaire
* Gérer le déséquilibre de classes
* Choisir des métriques adaptées
* Fournir ROC curve + AUC
* Justifier le seuil de décision. 

**Partie C – Interprétabilité**
* Identifier les variables les plus importantes
* Commenter le sens économique
* Repérer un biais potentiel (âge, genre, etc.). 

**Partie D – Qualité du code**
* Notebook structuré
* Fonctions claires
* Commentaires professionnels
* Reproductibilité

## 4. Grille d'évaluation

| Critère                       | Points  |
| ----------------------------- | ------- |
| Nettoyage des données propre  | 10      |
| Feature engineering pertinent | 15      |
| Pas de fuite de données       | 20      |
| Choix des métriques adapté    | 10      |
| Modèle bien entraîné          | 10      |
| Interprétabilité              | 10      |
| Analyse métier cohérente      | 15      |
| Qualité du code               | 10      |
| Visualisations claires        | 5       |
| Discussion sur biais/fairness | 5       |
| **Total**                     | **100** |


# 5. Niveau attendu par séniorité

**👶 Junior**
* Notebook propre
* Logistic regression OK
* AUC raisonnable
* Explications simples

**👨‍💻 Confirmé**
* CV propre
* Gestion du déséquilibre
* Feature engineering solide
* Interprétation claire

**🧠 Senior**
* Détection de data leakage
* Analyse fairness
* Pipeline propre
* Discussions gouvernance modèle
* Structuration en code modulaire

# 6. Question bonus très discriminante

« Votre modèle est performant mais le métier s’inquiète d’une discrimination sur l’âge. Que proposez-vous ? »  

Un bon candidat parlera de :
* fairness metrics
* biais indirect
* arbitrage performance / conformité
* documentation modèle

> Chargez les tables suivantes :

In [23]:
np.random.seed(0)

In [32]:
clients

,client_id,age,income,employment_status,seniority_years,gender,region
0,0,47,1729,unemployed,4,M,OUEST
1,1,19,907,CDD,4,F,OUEST
2,2,49,2572,unemployed,2,F,EST
3,3,62,2592,CDD,2,F,NORD
4,4,42,2031,unemployed,2,M,NORD
5,5,42,2739,freelance,4,F,EST
6,6,21,2678,CDI,4,F,OUEST
7,7,36,2848,CDD,3,F,OUEST
8,8,65,2463,CDD,3,F,EST
9,9,21,2540,unemployed,3,F,SUD


| colonne           | description               |
| ----------------- | ------------------------- |
| client_id         | identifiant               |
| loan_amount       | montant du prêt           |
| duration_months   | durée                     |
| interest_rate     | taux                      |
| previous_defaults | nombre d'incidents passés |
| default           | target                    |

In [39]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 3000

clients = pd.DataFrame({
    "client_id": range(n),
    "age": np.random.randint(20, 70, n),
    "income": np.random.normal(3000, 1000, n).clip(800),
    "employment_status": np.random.choice(["CDI", "CDD", "freelance", "unemployed"], n, p=[0.6,0.2,0.15,0.05]),
    "seniority_years": np.random.exponential(5, n),
    "gender": np.random.choice(["M","F"], n),
    "region": np.random.choice(["North","South","East","West"], n)
})

loans = pd.DataFrame({
    "client_id": range(n),
    "loan_amount": np.random.normal(15000, 5000, n).clip(2000),
    "duration_months": np.random.choice([12,24,36,48,60], n),
    "interest_rate": np.random.normal(4,1,n),
    "previous_defaults": np.random.poisson(0.3, n)
})

# True risk function (hidden)
risk = (
    0.3*(clients["age"] < 25) +
    0.4*(clients["income"] < 2000) +
    0.5*(loans["previous_defaults"] > 0) +
    0.2*(clients["employment_status"] == "unemployed")
)

# La sigmoide transforme le score en probabilité:
proba = 1/(1+np.exp(-risk))

>Reste à déterminer le défaut en fonction de cette probabilité.  
On va utiliser la loi de Bernoulli (stochastique réaliste) plutôt que proba > 0.5 alors défaut (déterministe pas de bruit => irréaliste).  
Pour rappel Bernoulli(0.3) exécuter 1000 fois donnera 300 fois 1 et 700 fois 0.

In [53]:
défaut = np.random.binomial(1,  proba * 0.6)

On a multiplié par 0.6 pour ajuster le taux global de défaut et qu'il soit réaliste (~20%).'

In [43]:
clients.isnull().mean()

client_id            0.0
age                  0.0
income               0.0
employment_status    0.0
seniority_years      0.0
gender               0.0
region               0.0
dtype: float64

In [50]:
clients[['age', 'income', 'seniority_years']].describe(include='all').transpose()

,count,mean,std,min,25%,50%,75%,max
age,3000.0,44.553000,14.334260,20.000000,32.000000,45.000000,56.000000,69.000000
income,3000.0,2994.860022,988.364931,800.000000,2322.030507,2995.532624,3662.714562,6926.237706
seniority_years,3000.0,4.936791,4.804545,0.000264,1.512284,3.478105,6.898346,40.130891


In [51]:
df = pd.merge(clients, loans, on='client_id')

In [ ]:
df.head()

,client_id,age,income,employment_status,seniority_years,gender,region,loan_amount,duration_months,interest_rate,previous_defaults,default
0,0,58,2448.141854,CDI,1.261253,M,South,14347.122714,36,3.759791,1,0
1,1,48,5558.199286,CDD,1.041635,F,South,17306.653565,24,4.931012,2,1
2,2,34,2435.752401,CDI,2.793251,F,East,12820.713225,24,4.493118,1,0
3,3,62,3184.551303,CDI,5.368873,F,South,17321.602804,24,4.955861,0,1
4,4,27,4542.109953,CDD,1.531309,M,East,17776.994303,48,3.190239,0,1
